<center><h1>Kalawadia_Krish_HW7</h1></center>

Name: Krish Kalawadia
<br>
Github Username: KK10-USC
<br>
USC ID: 4532974788

## 1. Multi-class and Multi-Label Classification Using Support Vector Machines

Import packages

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split


### (a) Download the Anuran Calls (MFCCs) Data Set

In [8]:
my_df = pd.read_csv("Frogs_MFCCs.csv")


training_set, testing_set = train_test_split(my_df,train_size = 0.7, random_state = 42)
print(training_set.shape)
print(testing_set.shape)

(5036, 26)
(2159, 26)


### (b) Train a classifier for each label

#### (i) Research

In [13]:
from sklearn.preprocessing import LabelEncoder,MultiLabelBinarizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, hamming_loss

labels = ["Family", "Genus","Species"]
features = []
for f in my_df.columns:
    if f not in labels:
        features.append(f)

x_train = training_set[features].values
x_test = testing_set[features].values

y_train = training_set[labels].values
y_test = testing_set[labels].values

raw_y_pred = np.empty_like(y_test,dtype=object)

for i, c in enumerate(labels):
    l = LabelEncoder()
    l.fit(my_df[c])

    y_train_col = l.transform(y_train[:,i])
    my_clf = OneVsRestClassifier(SVC(kernel = "rbf",random_state = 42))
    my_clf.fit(x_train,y_train_col)

    raw_y_pred[:,i] = l.inverse_transform(my_clf.predict(x_test))

y_test_list = []
y_pred_list = []

for r in y_test:
    y_test_list.append(list(r))

for r in raw_y_pred:
    y_pred_list.append(list(r))

mlb = MultiLabelBinarizer()
mlb.fit(y_test_list + y_pred_list)

y_true = mlb.transform(y_test_list)
y_pred = mlb.transform(y_pred_list)

exact_m = accuracy_score(y_true,y_pred)
h_loss = hamming_loss(y_true,y_pred)
ham_score = 1 - h_loss

print(f"Exact Match: {exact_m:.2f}")
print(f"Hamming Loss: {h_loss:.2f}")
print(f"Hamming Score: {ham_score:.2f}")


Exact Match: 0.83
Hamming Loss: 0.04
Hamming Score: 0.96


#### (ii) Train a SVM for each of the labels

In [16]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler

my_scaler = StandardScaler()
std_x_train = my_scaler.fit_transform(x_train)
std_x_test = my_scaler.transform(x_test)

my_param_grid = {
    "estimator__C" : [0.1,1,10,100],
    "estimator__gamma": [0.001,0.01,0.1,1]
}

my_cv = StratifiedKFold(n_splits = 10,shuffle = True, random_state = 42)
for i, c in enumerate(labels):
    l = LabelEncoder()
    l.fit(my_df[c])
    y_train_col = l.transform(y_train[:,i])

    orc = OneVsRestClassifier(SVC(kernel = "rbf",random_state = 42))
    my_grid = GridSearchCV(orc,my_param_grid,cv = my_cv,scoring = "accuracy",n_jobs = -1)
    my_grid.fit(x_train, y_train_col)

    raw_y_pred[:,i] = l.inverse_transform(my_grid.predict(x_test))

y_test_list = []
y_pred_list = []

for r in y_test:
    y_test_list.append(list(r))

for r in raw_y_pred:
    y_pred_list.append(list(r))

new_mlb = MultiLabelBinarizer()
new_mlb.fit(y_test_list + y_pred_list)

y_true = new_mlb.transform(y_test_list)
y_pred = new_mlb.transform(y_pred_list)

exact_m = accuracy_score(y_true,y_pred)
h_loss = hamming_loss(y_true,y_pred)
ham_score = 1 - h_loss

print(f"Exact Match: {exact_m:.2f}")
print(f"Hamming Loss: {h_loss:.2f}")
print(f"Hamming Score: {ham_score:.2f}")



Exact Match: 1.00
Hamming Loss: 0.00
Hamming Score: 1.00


#### (iii) Repeat 1(b)ii with L1-penalized SVMs

In [20]:
from sklearn.svm import LinearSVC
new_scaler = StandardScaler()
std_x_train = new_scaler.fit_transform(x_train)
std_x_test = new_scaler.transform(x_test)

my_param_g = {
    "estimator__C" : [0.001,0.01,0.1,1,10,100]
}
new_cv = StratifiedKFold(n_splits = 10, shuffle = True, random_state = 42)

for i,c in enumerate(labels):
    l = LabelEncoder()
    l.fit(my_df[c])
    y_train_col = l.transform(y_train[:,i])

    new_ocr = OneVsRestClassifier(LinearSVC(penalty = "l1", dual = False, max_iter = 50000, random_state = 42))
    new_grid = GridSearchCV(new_ocr, my_param_g, cv = new_cv, scoring = "accuracy",n_jobs = -1)
    new_grid.fit(std_x_train,y_train_col)

    raw_y_pred[:,i] = l.inverse_transform(new_grid.predict(std_x_test))


print(f"The Best L1-SVM Penalty for this dataset: {new_grid.best_params_["estimator__C"]}")


The Best L1-SVM Penalty for this dataset: 10


#### (iv) Repeat 1(b)iii by using SMOTE or any other method for imbalance

In [28]:
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

my_param_g = {
    "estimator__clf__C" : [0.001,0.01,0.1,1,10,100]
}
best_c = {}

for i,c in enumerate(labels):
    l = LabelEncoder()
    l.fit(my_df[c])
    y_train_col = l.transform(y_train[:,i])

    my_pipe = Pipeline([
    ("Smote",SMOTE(random_state = 42)),
    ("clf",LinearSVC(penalty = "l1",dual = False,max_iter = 50000,random_state = 42))
    ])

    another_orc = OneVsRestClassifier(my_pipe)
    another_grid = GridSearchCV(another_orc,my_param_g,cv = new_cv,scoring = "accuracy",n_jobs = -1)
    another_grid.fit(std_x_train,y_train_col)

    best_c[c] = another_grid.best_params_["estimator__clf__C"]
    print(f"The Best L1-SVM Penalty for this dataset: {best_c[c]}")





The Best L1-SVM Penalty for this dataset: 100
The Best L1-SVM Penalty for this dataset: 10
The Best L1-SVM Penalty for this dataset: 1


## 2. K-Means Clustering on a Multi-Class and Multi-Label Data Set

### (a) Use k-means clustering

In [31]:
from sklearn.metrics import silhouette_score
from sklearn.cluster import KMeans

new_df = pd.read_csv("Frogs_MFCCs.csv")
new_labels = ["Family","Genus","Species"]
new_feats = []
for f in new_df.columns:
    if f not in new_labels:
        new_feats.append(f)

x = new_df[new_feats].values
y = new_df[new_labels].values

k_scaler = StandardScaler()
std_x = k_scaler.fit_transform(x)

sil_scores = {}
for f in range(2,51):
    k = KMeans(n_clusters = f,n_init = 10,random_state = 42)
    labels = k.fit_predict(std_x)
    sil_scores[f] = silhouette_score(std_x,labels)

best_km = max(sil_scores, key = sil_scores.get)




### (b) Determine which family is the majority

In [33]:
final_km = KMeans(n_clusters = best_km,n_init = 10,random_state = 42)
cluster_labels = final_km.fit_predict(std_x)

cluster_df = pd.DataFrame(y, columns = new_labels)
cluster_df["Cluster"] = cluster_labels

family = {}
genus = {}
species = {}

for c in range(best_km):
    membs = cluster_df[cluster_df["Cluster"] == c]
    if len(membs) == 0:
        family[c] = None
        genus[c] = None
        species[c] = None
        continue

    family[c] = membs["Family"].mode()[0]
    genus[c] = membs["Genus"].mode()[0]
    species[c] = membs["Species"].mode()[0]
    

### (c) Calculate the average Hamming distance, Hamming score, and Hamming loss

In [34]:
assigned_y_cluster = np.empty_like(y,dtype = object)
for i,c in enumerate(cluster_labels):
    assigned_y_cluster[i,0] = family[c]
    assigned_y_cluster[i,1] = genus[c]
    assigned_y_cluster[i,2] = species[c]

list_of_y_true = []
list_of_y_pred = []

for f in y:
    list_of_y_true.append(f)

for f in assigned_y_cluster:
    list_of_y_pred.append(f)

another_mlb = MultiLabelBinarizer()
another_mlb.fit(list_of_y_true + list_of_y_pred)

y_true = another_mlb.transform(list_of_y_true)
y_pred = another_mlb.transform(list_of_y_pred)

ham_loss = hamming_loss(y_true,y_pred)
ham_score = 1 - ham_loss
exact_match = accuracy_score(y_true,y_pred)

print(f"Best K: {best_km}")
print(f"Avg Hamming Distance: {ham_loss:.3f}")
print(f"Hamming Score: {ham_score:.3f}")
print(f"Exact Match: {exact_match:.3f}")




Best K: 4
Avg Hamming Distance: 0.058
Hamming Score: 0.942
Exact Match: 0.728


## 3. ISLR 12.6.2

#### A)


#### Dissimilar Pairs
d(1,2) = 0.3\
d(1,3) = 0.4\
d(1,4) = 0.7\
d(2,3) = 0.5\
d(2,4) = 0.8\
d(3,4) = 0.45\

since the smallest distance is 0.3, we group 1 and 2 into one cluster.

#### using complete linkage
d-comp(A,B) = max(d(a,b)\
dcomp((1,2),3) = max((1,3),(2,3)) = max(0.4,0.5) = 0.5\
dcomp((1,2),4) = max((1,4),(2,4)) = max(0.7,0.8) = 0.8\
dcomp(3,4) = 0.45\

since the next smallest distance is 0.45, we group 3 and 4 into one cluster


now to find the max height of the diagram:

d((1,2),(3,4)) = max(d(1,3),d(2,3),d(1,4),d(2,4)) = max(0.4,0.7,0.5,0.8) = 0.8

so based on this math the splits start at 0.8, then 0.45, and then 0.3

#### Sketch
Put in github repo as another file

#### B)

#### using Single linkage
Again the first smallest distance is (1,2) because their distance is 0.3, so cluster 1 and 2 together

d-comp(A,B) = min(d(a,b)\
dcomp((1,2),3) = min((1,3),(2,3)) = min(0.4,0.5) = 0.4\
dcomp((1,2),4) = min((1,4),(2,4)) = min(0.7,0.8) = 0.7\
dcomp(3,4) = 0.45\

new clusters since new smallest is 0.4 --> new cluster = (1,2,3) , (4)\
min((1,2,3),(4) = min(d(1,4),d(2,4),d(3,4) = min(0.7,0.8,0.45) = 0.45\
height = 0.45


#### Sketch
Put in github repo as another file.

#### C)

Observations 1 and 2 are in one cluster, while observations 3 and 4 are in another separate cluster based on the dendrogram.

#### D)

Observations 1,2, and 3 are in one cluster and observation 4 will be in a separate cluster. 

#### E)